# WP008 — Market benchmark: WP007's models vs. the bookmaker line

Re-runs WP003's benchmark (model vs. Pinnacle/Bet365/market-average, de-vigged) on WP007's larger, paired dataset (1,314 matches vs. 401) and on two arms: `L0_baseline` (the WP001-validated config) and `L5_xgchannel` (WP007's one measured improvement). The question this answers: does the xG-channel gain actually close any of the gap to the market, or is it orthogonal to it?

Odds come from a GitHub mirror of football-data.co.uk (see `fetch_odds.py`'s docstring for why — the original site is blocked by network policy on the machine this ran on) — verified before use: team names match Understat's spelling exactly, and Pinnacle/Bet365/market-average coverage was cross-checked against the expected match counts.

In [1]:
import sys, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from market_analysis import load_wp007_matches, join_market, add_devig_probs, book_rps, BOOKS
sys.path.insert(0, '../wp007_structural_fixes')
from analysis import bootstrap_mean_ci, paired_diff, verdict, reliability_table, rps_hda

pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

ARMS = ['L0_baseline', 'L5_xgchannel']
M = load_wp007_matches(ARMS)
odds = pd.read_pickle('odds_raw.pkl')
print(f'{M.arm.nunique()} arms x {M[M.arm==ARMS[0]].shape[0]} matches loaded from WP007')

joined, unmatched, mismatched = join_market(M[M.arm == ARMS[0]], odds)  # arm-independent identity + odds
print(f'{len(joined)}/{len(joined)+len(unmatched)+len(mismatched)} matched to the odds source '
      f'({len(unmatched)} unmatched, {len(mismatched)} score mismatches dropped)')
for book in BOOKS:
    print(f'  {book}: {joined[f"{book}_h"].notna().sum()}/{len(joined)} have closing odds, '
         f'mean overround {joined[f"overround_{book}"].mean():.2%}')

2 arms x 1314 matches loaded from WP007
1314/1314 matched to the odds source (0 unmatched, 0 score mismatches dropped)
  pinnacle: 1205/1314 have closing odds, mean overround 2.68%
  bet365: 1314/1314 have closing odds, mean overround 5.48%
  market_avg: 1314/1314 have closing odds, mean overround 4.49%


## 1. Pooled RPS — naive vs. both model arms vs. the market

Primary comparison restricted to the Pinnacle subset (matches WP003's approach — Pinnacle is the sharpest book and the one prior work benchmarked against); Bet365/market-average cover every matched fixture and are shown for context.

In [2]:
key = ['window', 'round', 'home', 'away']
pin_ok = joined.set_index(key)[['pinnacle_h','pinnacle_d','pinnacle_a']].notna().all(axis=1)
pin_keys = pin_ok[pin_ok].index

naive_h, naive_a = M['goals_home'].mean(), M[M.arm==ARMS[0]]['goals_home'].mean(),
from football_model.model.predict import dc_outcome_probs
import pickle
df_cv = pickle.load(open('../wp007_structural_fixes/cv_shared_data.pkl','rb'))['df_cv']
naive_p = np.array(dc_outcome_probs(df_cv.goals_home.mean(), df_cv.goals_away.mean()))

rows = []
for arm in ARMS:
    Ma = M[M.arm == arm].set_index(key).loc[pin_keys]
    r = rps_hda(Ma[['p_home','p_draw','p_away']].to_numpy(), Ma['result'])
    g = Ma.index.get_level_values('window')
    m, lo, hi = bootstrap_mean_ci(r, groups=g)
    rows.append({'predictor': arm, 'n': len(r), 'pooled_rps': round(m, 4), 'CI (cluster)': f'[{lo:.4f}, {hi:.4f}]'})

Jp = joined.set_index(key).loc[pin_keys].reset_index()
for book in BOOKS:
    r, sub = book_rps(Jp, book)
    g = sub.set_index(key).index.get_level_values('window') if False else sub['window']
    m, lo, hi = bootstrap_mean_ci(r, groups=sub['window'].to_numpy())
    rows.append({'predictor': book, 'n': len(r), 'pooled_rps': round(m, 4), 'CI (cluster)': f'[{lo:.4f}, {hi:.4f}]'})

naive_rps = rps_hda(np.tile(naive_p, (len(pin_keys), 1)), M[M.arm==ARMS[0]].set_index(key).loc[pin_keys, 'result'])
m, lo, hi = bootstrap_mean_ci(naive_rps)
rows.append({'predictor': 'naive', 'n': len(naive_rps), 'pooled_rps': round(m, 4), 'CI (cluster)': f'[{lo:.4f}, {hi:.4f}]'})

print(f'n = {len(pin_keys)} matches with Pinnacle closing odds\n')
print(pd.DataFrame(rows).to_string(index=False))

n = 1205 matches with Pinnacle closing odds

   predictor    n  pooled_rps     CI (cluster)
 L0_baseline 1205      0.2053 [0.1955, 0.2146]
L5_xgchannel 1205      0.2027 [0.1930, 0.2120]
    pinnacle 1205      0.1972 [0.1867, 0.2071]
      bet365 1205      0.1975 [0.1870, 0.2072]
  market_avg 1205      0.1987 [0.1886, 0.2083]
       naive 1205      0.2378 [0.2346, 0.2407]


## 2. Paired gap to Pinnacle — does the xG channel actually close it?

Same matches, matched fixture-for-fixture. Negative = model beats Pinnacle; positive = model worse. This is the number WP004/WP005 chased and couldn't move — does WP007's one validated change do any better?

In [3]:
rows = []
for arm in ARMS:
    Ma = M[M.arm == arm].set_index(key).loc[pin_keys]
    r_model = rps_hda(Ma[['p_home','p_draw','p_away']].to_numpy(), Ma['result'])
    r_pin, _ = book_rps(Jp, 'pinnacle')
    diff = r_model - r_pin
    g = Ma.index.get_level_values('window')
    m, lo, hi = bootstrap_mean_ci(diff, groups=g)
    rows.append({'arm': arm, 'gap vs Pinnacle': round(m, 4), 'CI (cluster)': f'[{lo:+.4f}, {hi:+.4f}]', 'verdict': verdict(lo, hi)})
gaps = pd.DataFrame(rows)
print(gaps.to_string(index=False))

g0, g1 = gaps.loc[gaps.arm==ARMS[0], 'gap vs Pinnacle'].iloc[0], gaps.loc[gaps.arm==ARMS[1], 'gap vs Pinnacle'].iloc[0]
if g0 != 0:
    print(f'\n{ARMS[1]} closes {(1 - g1/g0):.0%} of {ARMS[0]}\'s gap to Pinnacle' if g0 > 0 else '')

# is the DIFFERENCE between the two arms' gaps itself distinguishable from zero?
M0 = M[M.arm==ARMS[0]].set_index(key).loc[pin_keys]; M1 = M[M.arm==ARMS[1]].set_index(key).loc[pin_keys]
r0 = rps_hda(M0[['p_home','p_draw','p_away']].to_numpy(), M0['result'])
r1 = rps_hda(M1[['p_home','p_draw','p_away']].to_numpy(), M1['result'])
d, lo, hi = paired_diff(r1, r0, groups=M0.index.get_level_values('window'))
print(f'\n{ARMS[1]} vs {ARMS[0]} directly (not via Pinnacle): {d:+.4f} [{lo:+.4f}, {hi:+.4f}] {verdict(lo, hi)}')

         arm  gap vs Pinnacle       CI (cluster) verdict
 L0_baseline           0.0081 [+0.0045, +0.0115]   worse
L5_xgchannel           0.0055 [+0.0023, +0.0087]   worse

L5_xgchannel closes 32% of L0_baseline's gap to Pinnacle

L5_xgchannel vs L0_baseline directly (not via Pinnacle): -0.0026 [-0.0048, -0.0003] better


## 3. Calibration vs. Pinnacle

In [4]:
for arm in ARMS:
    Ma = M[M.arm == arm].set_index(key).loc[pin_keys]
    print(f'{arm} — predicted P(home win) vs actual:')
    print(reliability_table(Ma['p_home'], (Ma['result']=='H')).to_string(index=False), '\n')
print('Pinnacle — predicted P(home win) vs actual:')
print(reliability_table(Jp['pinnacle_h'], (Jp['result']=='H')).to_string(index=False))

L0_baseline — predicted P(home win) vs actual:
      bin   n  pred  actual
0.00-0.12  29 0.102   0.069
0.12-0.25 148 0.198   0.135
0.25-0.38 289 0.317   0.332
0.38-0.50 315 0.433   0.438
0.50-0.62 243 0.560   0.551
0.62-0.75 127 0.679   0.669
0.75-0.88  52 0.788   0.885
0.88-1.00   2 0.883   1.000 

L5_xgchannel — predicted P(home win) vs actual:
      bin   n  pred  actual
0.00-0.12  36 0.093   0.056
0.12-0.25 142 0.197   0.190
0.25-0.38 275 0.315   0.291
0.38-0.50 297 0.434   0.431
0.50-0.62 232 0.557   0.539
0.62-0.75 156 0.681   0.654
0.75-0.88  64 0.798   0.875
0.88-1.00   3 0.895   1.000 

Pinnacle — predicted P(home win) vs actual:
      bin   n  pred  actual
0.00-0.12  58 0.098   0.034
0.12-0.25 177 0.193   0.198
0.25-0.38 278 0.313   0.320
0.38-0.50 238 0.434   0.429
0.50-0.62 224 0.563   0.567
0.62-0.75 144 0.678   0.653
0.75-0.88  79 0.802   0.861
0.88-1.00   7 0.892   0.857


## 4. Where the model disagrees most with the market

In [5]:
d = Jp.copy()
d['disagreement'] = (M[M.arm==ARMS[1]].set_index(key).loc[pin_keys, 'p_home'].to_numpy() - d['pinnacle_h']).abs()
d['rps_model'] = rps_hda(M[M.arm==ARMS[1]].set_index(key).loc[pin_keys, ['p_home','p_draw','p_away']].to_numpy(), d['result'])
r_pin, _ = book_rps(Jp, 'pinnacle')
d['rps_mkt'] = r_pin
rows = []
for label, q in [('all matches', 0.0), ('top 50% disagreement', 0.5), ('top 25%', 0.75), ('top 10%', 0.90)]:
    sub = d[d['disagreement'] >= d['disagreement'].quantile(q)]
    gap = sub['rps_model'].mean() - sub['rps_mkt'].mean()
    _, lo, hi = bootstrap_mean_ci((sub['rps_model'] - sub['rps_mkt']).to_numpy(), groups=sub['window'].to_numpy())
    rows.append({'subset': label, 'n': len(sub), f'RPS {ARMS[1]}': round(sub.rps_model.mean(),4),
                 'RPS Pinnacle': round(sub.rps_mkt.mean(),4), 'gap': round(gap,4), 'CI (cluster)': f'[{lo:+.4f}, {hi:+.4f}]'})
print(pd.DataFrame(rows).to_string(index=False))

              subset    n  RPS L5_xgchannel  RPS Pinnacle    gap       CI (cluster)
         all matches 1205            0.2027        0.1972 0.0055 [+0.0023, +0.0087]
top 50% disagreement  603            0.2144        0.2032 0.0112 [+0.0055, +0.0169]
             top 25%  302            0.2161        0.1993 0.0168 [+0.0082, +0.0253]
             top 10%  121            0.2343        0.2085 0.0257 [+0.0094, +0.0408]


## 5. Season-by-season stability of the market gap

In [6]:
rows = []
for s, idx in Jp.groupby('season').groups.items():
    sub_j = Jp.loc[idx]
    keys_s = sub_j.set_index(key).index
    Ma = M[M.arm==ARMS[1]].set_index(key).reindex(keys_s).dropna(subset=['p_home'])
    if len(Ma) == 0: continue
    r_model = rps_hda(Ma[['p_home','p_draw','p_away']].to_numpy(), Ma['result'])
    r_pin, _ = book_rps(sub_j.set_index(key).loc[Ma.index].reset_index(), 'pinnacle')
    diff = r_model - r_pin
    g = Ma.index.get_level_values('window')
    m, lo, hi = bootstrap_mean_ci(diff, groups=g)
    rows.append({'season': s, 'n': len(Ma), 'gap vs Pinnacle': round(m,4), 'CI (cluster)': f'[{lo:+.4f}, {hi:+.4f}]'})
print(pd.DataFrame(rows).sort_values('season').to_string(index=False))

season   n  gap vs Pinnacle       CI (cluster)
  2020 223           0.0074 [+0.0038, +0.0121]
  2021 220          -0.0001 [-0.0078, +0.0066]
  2022 205           0.0089 [+0.0012, +0.0156]
  2023 228           0.0112 [+0.0041, +0.0191]
  2024 219          -0.0008 [-0.0079, +0.0074]
  2025 110           0.0072 [+0.0012, +0.0145]
